# Run training manifests from an allocated Jupyter session

This notebook bypasses `sbatch` and runs the same treatment-anchor event-level Python entry points directly. It is resumable: completed outputs are skipped whenever the task list is rebuilt. Each task writes to its own log file, and one failed event does not stop the remaining events.

**Run this only in a Jupyter session backed by an allocated compute node—not on an ERISTwo login node.** Start with the `full_cohort` workflow. Run `full_cohort_risk_scores` only after full-cohort training is complete.

In [ ]:
from __future__ import annotations

import json
import os
import signal
import socket
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

from tqdm.auto import tqdm


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.py').is_file() and (candidate / 'pipelines').is_dir():
            return candidate
    raise RuntimeError(f'Could not find v2 root from {start}')


V2_ROOT = find_v2_root()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from config import RESULTS_PATH
from schemes import get_output_dir

print(f'Python:  {sys.executable}')
print(f'Host:    {socket.gethostname()}')
print(f'v2 root: {V2_ROOT}')


## Configuration

The defaults target a 4-CPU/128-GB Jupyter allocation: four events run concurrently with one model thread each. This favors throughput across the large manifest and uses the available memory to avoid serial event processing.

In [ ]:
# This fallback notebook is intentionally restricted to treatment-anchor runs.
ANCHOR = 'treatment'
WORKFLOW = 'full_cohort'  # full_cohort | feature_comp | full_cohort_risk_scores

# Speed-oriented defaults for this notebook's 4-CPU/128-GB allocation. Running four
# independent events keeps all cores occupied and avoids nested joblib overhead.
ALLOCATED_CPUS = int(os.environ.get('SLURM_CPUS_PER_TASK', '4'))
N_JOBS = 1
MAX_CONCURRENT_TASKS = ALLOCATED_CPUS
MAX_ITER = 2500
BACKEND = 'threading'
OVERWRITE = False

# Optional bounds for a trial/chunk. Leave both as shown for every pending row.
START_AT = 0
MAX_TASKS = None  # e.g. 5 for a short trial

VALID_WORKFLOWS = {'full_cohort', 'feature_comp', 'full_cohort_risk_scores'}
if WORKFLOW not in VALID_WORKFLOWS:
    raise ValueError(f'WORKFLOW must be one of {sorted(VALID_WORKFLOWS)}')
if ANCHOR != 'treatment':
    raise ValueError('This notebook is restricted to the treatment anchor')
if min(ALLOCATED_CPUS, N_JOBS, MAX_CONCURRENT_TASKS) < 1:
    raise ValueError('CPU and concurrency values must be positive')
if N_JOBS * MAX_CONCURRENT_TASKS > ALLOCATED_CPUS:
    raise ValueError('N_JOBS * MAX_CONCURRENT_TASKS exceeds ALLOCATED_CPUS')

suffix = '' if ANCHOR == 'treatment' else f'__{ANCHOR}'
manifest_stem = 'feature_comp_tasks' if WORKFLOW == 'feature_comp' else 'full_cohort_tasks'
MANIFEST = V2_ROOT / 'slurm' / 'slurm_manifests' / f'{manifest_stem}{suffix}.tsv'

print(f'Workflow:       {WORKFLOW}')
print(f'Anchor:         {ANCHOR}')
print(f'Manifest:       {MANIFEST}')
print(f'Allocated CPUs: {ALLOCATED_CPUS}')
print(f'Model n_jobs:   {N_JOBS}')
print(f'Concurrent:     {MAX_CONCURRENT_TASKS}')


## Build the pending task list

The completion checks match the files expected by the manifest builder and worker scripts. Rerun this cell after an interrupted session to remove completed tasks.

In [ ]:
SCHEME_ALIASES = {'icd3': 'icd3_post', 'icd4': 'icd4_post', 'phecode': 'phecode_post'}
MODALITIES = ['stage', 'treatment', 'somatic', 'prs', 'text', 'metburden']
FULL_FILES = [
    'text_test.csv', 'text_val.csv', 'text_ipcw_reference.csv.gz',
    'base_test.csv', 'base_val.csv', 'base_ipcw_reference.csv.gz',
]


def read_manifest(path: Path) -> list[tuple[str, str]]:
    if not path.is_file():
        raise FileNotFoundError(f'Manifest not found: {path}')
    rows = []
    for line_number, raw in enumerate(path.read_text().splitlines(), 1):
        if not raw.strip():
            continue
        fields = raw.split('\t')
        if len(fields) < 2 or len(fields) > 3:
            raise ValueError(f'{path}:{line_number}: expected 2 or 3 tab-separated fields')
        scheme, event = fields[:2]
        if not scheme or not event:
            raise ValueError(f'{path}:{line_number}: scheme and event are required')
        rows.append((SCHEME_ALIASES.get(scheme, scheme), event))
    return rows


def full_cohort_done(scheme: str, event: str) -> bool:
    out = Path(get_output_dir(scheme, 'full_cohort', ANCHOR)) / event
    return all((out / name).is_file() for name in FULL_FILES)


def feature_comp_done(scheme: str, event: str) -> bool:
    out = Path(get_output_dir(scheme, 'feature_comps', ANCHOR)) / event
    risk = out.parent.parent / 'held_out_risk_scores' / event
    grid_files = [out / f'{mod}_{kind}.csv' for mod in MODALITIES for kind in ('test', 'val')]
    audit_files = [out / f'{mod}_ipcw_reference.csv.gz' for mod in MODALITIES]
    risk_files = [risk / f'{mod}_risk_scores.csv' for mod in MODALITIES]
    return all(path.is_file() for path in grid_files + audit_files + risk_files)


def full_risk_done(scheme: str, event: str) -> bool:
    out = Path(get_output_dir(scheme, 'full_cohort_risk_scores', ANCHOR)) / event
    return all((out / name).is_file() for name in ('text_risk_scores.csv', 'base_risk_scores.csv'))


all_tasks = read_manifest(MANIFEST)
if WORKFLOW == 'full_cohort':
    pending = [task for task in all_tasks if OVERWRITE or not full_cohort_done(*task)]
elif WORKFLOW == 'feature_comp':
    pending = [task for task in all_tasks if OVERWRITE or not feature_comp_done(*task)]
else:
    eligible = [task for task in all_tasks if full_cohort_done(*task)]
    pending = [task for task in eligible if OVERWRITE or not full_risk_done(*task)]
    print(f'Risk-score eligibility: {len(eligible)}/{len(all_tasks)} full-cohort tasks complete')

tasks = pending[START_AT:]
if MAX_TASKS is not None:
    tasks = tasks[:MAX_TASKS]

print(f'Manifest rows: {len(all_tasks)}')
print(f'Already done/ineligible: {len(all_tasks) - len(pending)}')
print(f'Pending selected for this run: {len(tasks)}')
print('First tasks:', tasks[:5])


## Run selected tasks

A timestamped run directory under the shared results path stores one log per event plus a JSON-lines status ledger. Output is streamed directly to disk, so long jobs do not inflate notebook memory. Interrupting the notebook stops queueing results; rerunning the task-list cell later resumes from output completeness.

In [ ]:
MODULES = {
    'full_cohort': 'pipelines.training.run_full_cohort_event',
    'feature_comp': 'pipelines.training.run_feature_comp_task',
    'full_cohort_risk_scores': 'pipelines.training.run_full_cohort_risk_scores',
}
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = Path(RESULTS_PATH) / 'notebook_runs' / ANCHOR / WORKFLOW / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
LEDGER = RUN_DIR / 'status.jsonl'

child_env = os.environ.copy()
child_env.update({
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1',
    'NUMEXPR_NUM_THREADS': '1',
})


def build_command(scheme: str, event: str) -> list[str]:
    cmd = [
        sys.executable, '-m', MODULES[WORKFLOW],
        '--scheme', scheme, '--event', event, '--anchor', ANCHOR,
        '--n-jobs', str(N_JOBS), '--max-iter', str(MAX_ITER), '--backend', BACKEND,
    ]
    if WORKFLOW == 'feature_comp':
        cmd += ['--modality', 'all']
    if OVERWRITE:
        cmd.append('--overwrite')
    return cmd


def run_task(task: tuple[str, str]) -> dict:
    scheme, event = task
    safe_event = ''.join(c if c.isalnum() or c in '._-' else '_' for c in event)
    log_path = RUN_DIR / f'{scheme}__{safe_event}.log'
    started = time.time()
    with log_path.open('w', encoding='utf-8') as log:
        log.write('COMMAND: ' + ' '.join(build_command(scheme, event)) + '\n\n')
        log.flush()
        proc = subprocess.Popen(
            build_command(scheme, event), cwd=V2_ROOT, env=child_env,
            stdout=log, stderr=subprocess.STDOUT, text=True, start_new_session=True,
        )
        try:
            returncode = proc.wait()
        except KeyboardInterrupt:
            # Stop the complete joblib process group before returning control to Jupyter.
            os.killpg(proc.pid, signal.SIGTERM)
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait()
            raise
    return {
        'scheme': scheme, 'event': event, 'returncode': returncode,
        'elapsed_minutes': round((time.time() - started) / 60, 2),
        'log': str(log_path),
    }


def record_result(result: dict) -> None:
    results.append(result)
    with LEDGER.open('a', encoding='utf-8') as ledger:
        ledger.write(json.dumps(result) + '\n')
    if result['returncode'] != 0:
        print(f"[FAIL] {result['scheme']}:{result['event']} -> {result.get('log', result.get('error'))}")


results = []
if MAX_CONCURRENT_TASKS == 1:
    # Keep the default path in the notebook's main thread so Interrupt Kernel stops the
    # current subprocess instead of waiting behind a pre-submitted queue.
    for task in tqdm(tasks, desc=WORKFLOW):
        record_result(run_task(task))
else:
    # Submit only one bounded batch at a time. An interrupt can wait for at most the
    # currently running batch, never for every remaining manifest row.
    progress = tqdm(total=len(tasks), desc=WORKFLOW)
    for offset in range(0, len(tasks), MAX_CONCURRENT_TASKS):
        batch = tasks[offset:offset + MAX_CONCURRENT_TASKS]
        with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_TASKS) as pool:
            futures = {pool.submit(run_task, task): task for task in batch}
            for future in as_completed(futures):
                try:
                    result = future.result()
                except Exception as exc:
                    scheme, event = futures[future]
                    result = {'scheme': scheme, 'event': event, 'returncode': -1, 'error': repr(exc)}
                record_result(result)
                progress.update(1)
    progress.close()

n_ok = sum(result['returncode'] == 0 for result in results)
print(f'Finished: {n_ok} succeeded, {len(results) - n_ok} failed')
print(f'Run logs: {RUN_DIR}')


## Inspect failures and resume

Inspect the listed log files. After resolving any issue, rerun **Build the pending task list** and **Run selected tasks**; completed events will be excluded automatically.

In [ ]:
failures = [result for result in results if result['returncode'] != 0]
print(f'Failures: {len(failures)}')
for result in failures[:50]:
    print(f"  {result['scheme']}:{result['event']}  {result.get('log', result.get('error'))}")
if len(failures) > 50:
    print(f'  ... and {len(failures) - 50} more; see {LEDGER}')
